# Silver Layer – Customer Information

This notebook transforms the customer data from the Bronze layer into the Silver layer by applying data cleaning and standardization rules.

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:
df = spark.read.table('workspace.bronze.crm_cust_info')


## Remove Duplicate Customer IDs

Keep the latest record for each customer ID and remove null IDs.

In [0]:
w = Window.partitionBy('cst_id').orderBy(desc('cst_create_date'))
df = (df.withColumn('rn' ,  row_number().over(w) )
      .filter(col('rn') ==  1)
      .filter(col('cst_id').isNotNull() )
      .drop('rn')    )

## Remove Duplicate Customer Keys

Keep the latest record for each customer key and remove null keys.

In [0]:
w = Window.partitionBy('cst_key').orderBy(desc('cst_create_date'))
df = (df.withColumn('rn' ,  row_number().over(w) )
      .filter(col('rn') ==  1)
      .filter(col('cst_key').isNotNull() )
      .drop('rn')    )

## Trim Text Fields

Remove leading and trailing spaces from customer names.

In [0]:
df = (
    df.withColumn('cst_firstname' , trim(col('cst_firstname')))
    .withColumn('cst_lastname' , trim(col('cst_lastname')))
    )

## Standardize Marital Status

Convert marital status codes to descriptive values.

In [0]:
df  = df.withColumn('cst_marital_status' , 
              when(col("cst_marital_status" )  == 'S' , 'Single')
              .when(col("cst_marital_status" )  == 'M' , 'Married')
              .otherwise('N/A')
              )

## Standardize Gender

Convert gender codes to descriptive values.

In [0]:
df  = df.withColumn('cst_gndr' , 
              when(col("cst_gndr") =='M'  , 'Male')
              .when(col("cst_gndr") =='F'  , 'Female')
              .otherwise('N/A')
              )

## Rename Columns

Rename source columns to standardized business-friendly names.

In [0]:
rename_map = {
    'cst_id' : 'customer_id',
    'cst_key': 'customer_number' ,
    'cst_firstname' :'first_name' ,
    'cst_lastname' : 'last_name' ,
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name , new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

## Preview Transformed Data

Review the transformed dataset before loading it into the Silver layer.

In [0]:
df.limit(10).display()

## Load to Silver Layer

Write the cleaned and standardized customer data to the Silver layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.crm_cust_info')

## Verify Silver Table

Validate that the transformed data has been successfully loaded into the Silver layer.

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_cust_info LIMIT 10